# HATC-IDS — Complete Pipeline, Results and Hardware Validation

**Datasets: Edge-IIoTset and CIC-IIoT-2025.**

This single notebook does three things:

| part | cells | needs |
|---|---|---|
| **A. Pipeline** — prep → training → export | 1–6 | GPU (skipped if outputs exist) |
| **B. Results** — tables, Pareto plots, per-class | 7–11 | nothing, reads `results/` |
| **C. Hardware** — parity, latency, energy | 12–15 | ESP32-S3 + arduino-cli |

**Resumable.** Every pipeline stage checks its own outputs and skips if present.
As shipped, this folder is warm: Part A skips entirely and the notebook runs in
about two minutes. Set `FORCE = True` to re-run regardless.

**Safe without hardware.** Part C auto-detects the board and no-ops if none is
attached.

## 0 — Configuration and environment

In [ ]:
import os, sys, subprocess, glob, time, json
import numpy as np, pandas as pd

HERE = os.getcwd()
PY   = sys.executable
CLI  = os.path.expanduser("~/tools/arduino-cli/arduino-cli.exe")

# ---- knobs -----------------------------------------------------------------
DATASETS   = ["edge", "cic25"]     # both reported corpora
FORCE      = False                 # True = re-run stages even if outputs exist
BOARD_PORT = None                  # e.g. "COM6"; None = auto-detect
SEEDS      = "3"
# ----------------------------------------------------------------------------

CACHE  = {"edge": "edge_full.npz",  "cic25": "cic25_full.npz"}
LABEL  = {"edge": "Edge-IIoTset",   "cic25": "CIC-IIoT-2025"}

os.makedirs("results", exist_ok=True); os.makedirs("logs", exist_ok=True)

def have(p): return os.path.exists(p) and os.path.getsize(p) > 0

def run(cmd, log=None, timeout=None, env=None):
    print(f"$ {' '.join(cmd) if isinstance(cmd,list) else cmd}")
    t0 = time.time()
    r = subprocess.run(cmd, shell=isinstance(cmd,str), capture_output=True,
                       text=True, timeout=timeout, env=env)
    out = (r.stdout or "") + (r.stderr or "")
    if log: open(os.path.join("logs", log), "w", encoding="utf-8").write(out)
    print(f"  rc={r.returncode}  {time.time()-t0:.0f}s")
    tail = "\n".join(out.strip().split("\n")[-10:])
    if tail: print(tail)
    return r.returncode, out

def stage(name, outputs, fn):
    outs = [outputs] if isinstance(outputs,str) else outputs
    if not FORCE and all(have(o) for o in outs):
        print(f"[SKIP] {name}"); return False
    print(f"[RUN ] {name}"); fn(); return True

print("python :", PY)
print("datasets:", ", ".join(LABEL[d] for d in DATASETS))

In [ ]:
import importlib
for m in ["numpy","pandas","torch","sklearn","matplotlib","imblearn","docx"]:
    try:
        mod = importlib.import_module(m)
        print(f"  OK   {m:12s} {getattr(mod,'__version__','?')}")
    except ImportError:
        print(f"  MISS {m:12s} <- pip install")

import torch
print("\ncuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("arduino-cli:", "found" if os.path.exists(CLI) else "MISSING")
print("\ncaches:")
for d in DATASETS:
    f = CACHE[d]
    print(f"  {'OK  ' if have(f) else 'MISS'} {f}" + (f"  ({os.path.getsize(f)/1e6:.1f} MB)" if have(f) else ""))

---
# Part A — Pipeline

Skipped automatically when outputs already exist. A cold run of the training
stages is roughly 4–6 h of GPU time; data prep additionally requires the raw
corpora, which are **not** in this folder.

### 1 — Data preparation

In [ ]:
def _prep():
    if "edge" in DATASETS and not have(CACHE["edge"]):
        run([PY,"scripts/prep_edge.py"], "prep_edge.log", timeout=7200)
    if "cic25" in DATASETS and not have(CACHE["cic25"]):
        run([PY,"scripts/prep_cic25.py"], "prep_cic25.log", timeout=7200)

stage("data prep", [CACHE[d] for d in DATASETS], _prep)

### 2 — Accuracy frontier sweeps

In [ ]:
def _frontier():
    for d in DATASETS:
        out = f"results/final_results_{d}.csv"
        if have(out) and not FORCE: continue
        env = dict(os.environ, HATC_CACHE=os.path.join(HERE,CACHE[d]),
                   HATC_OUT=out, HATC_SEEDS=SEEDS, HATC_WIDE="0")
        print(f"  -> {out}")
        run([PY,"scripts/final_experiment.py"], f"frontier_{d}.log", timeout=14400, env=env)

stage("frontier sweeps", [f"results/final_results_{d}.csv" for d in DATASETS], _frontier)

### 3 — INT8 quantisation and LCT-IDS baseline

In [ ]:
def _qat():
    for d in DATASETS:
        out = f"results/qat_results_{d}.csv"
        if have(out) and not FORCE: continue
        env = dict(os.environ, HATC_CACHE=os.path.join(HERE,CACHE[d]), QAT_OUT=out,
                   HATC_OUT=f"results/final_results_{d}.csv", HATC_SEEDS=SEEDS)
        run([PY,"scripts/qat_int8.py"], f"qat_{d}.log", timeout=14400, env=env)

def _lct():
    for d in DATASETS:
        out = f"results/lct_results_{d}.csv"
        if have(out) and not FORCE: continue
        env = dict(os.environ, HATC_CACHE=os.path.join(HERE,CACHE[d]),
                   LCT_OUT=out, HATC_SEEDS=SEEDS)
        print(f"  -> {out}  (slow: ~15-25 min/seed)")
        run([PY,"scripts/lct_ids.py"], f"lct_{d}.log", timeout=21600, env=env)

stage("INT8 PTQ/QAT", [f"results/qat_results_{d}.csv" for d in DATASETS], _qat)
stage("LCT-IDS",      [f"results/lct_results_{d}.csv" for d in DATASETS], _lct)

### 4 — Per-class analysis and logit calibration

In [ ]:
stage("per-class analysis",
      ["results/confusion_matrices.csv","results/confusion_analysis.txt"],
      lambda: run([PY,"scripts/confusion_analysis.py"], "confusion_analysis.log", timeout=14400))

### 5 — Firmware export (all frontier configs, both datasets)

In [ ]:
def _export():
    for d in DATASETS:
        run([PY,"scripts/export_configs.py",CACHE[d]], f"export_{d}.log", timeout=14400)

stage("firmware export",
      [f"firmware/configs/{'edge' if d=='edge' else 'cic25'}_hatc_d24_k8/hatc_weights.h"
       for d in DATASETS],
      _export)

print("\nexported configs:")
for p in sorted(glob.glob("firmware/configs/*")):
    if not p.endswith("_original"): print("  ", os.path.basename(p))

### 6 — Consolidate + rebuild the .docx report

In [ ]:
run([PY,"scripts/build_consolidated.py"], "build_consolidated.log")
run([PY,"scripts/build_report.py"], "build_report.log")
print("\nHATC-IDS_Report.docx:", "OK" if have("HATC-IDS_Report.docx") else "MISSING")

---
# Part B — Results

Reads `results/`. Runs without a GPU or a board.

### 7 — Accuracy frontier

In [ ]:
df = pd.read_csv("results/consolidated_results.csv")
for ds in df.dataset.unique():
    print("="*80); print(ds); print("="*80)
    d = df[df.dataset==ds].sort_values("macro_f1", ascending=False)
    print(d[["arm","family","macro_f1","macro_f1_sd","minority_recall",
             "analytic_macs","flash_int8_bytes"]].to_string(index=False))
    print()

### 8 — Measured on-device results

Median of 256 inferences, cache-cold first call excluded. `us_per_mac` is the
diagnostic: HATC's gather/accumulate costs more per MAC than the MLP's dense
GEMM, which is why analytic MAC counts mispredict latency here.

In [ ]:
m = df[df.measured_latency_us.notna()].copy()
m["us_per_mac"] = m.measured_latency_us / m.analytic_macs
for ds in m.dataset.unique():
    print("="*80); print(ds); print("="*80)
    print(m[m.dataset==ds].sort_values("measured_latency_us")[
        ["arm","macro_f1","measured_latency_us","analytic_macs",
         "flash_int8_bytes","us_per_mac","measured_parity"]].to_string(index=False))
    print()

### 9 — The measured Pareto frontier

In [ ]:
import matplotlib.pyplot as plt

def pareto(points):
    return [lab for lab,us,f1 in points
            if not any((o<=us and g>=f1 and (o<us or g>f1)) for l,o,g in points if l!=lab)]

fig, axes = plt.subplots(1, len(m.dataset.unique()), figsize=(13,5), squeeze=False)
for ax, ds in zip(axes[0], m.dataset.unique()):
    d = m[m.dataset==ds]
    for fam, mk, col in [("HATC-v3","o","#2b6cb0"), ("MLP","s","#c05621")]:
        s = d[d.family==fam]
        ax.scatter(s.measured_latency_us, s.macro_f1, marker=mk, s=100,
                   label=fam, color=col, zorder=3)
        for _, r in s.iterrows():
            ax.annotate(r.arm.replace("HATC-v3 ","").replace("MLP ",""),
                        (r.measured_latency_us, r.macro_f1),
                        textcoords="offset points", xytext=(6,4), fontsize=8)
    pts = list(zip(d.arm, d.measured_latency_us, d.macro_f1))
    fr = sorted([p for p in pts if p[0] in pareto(pts)], key=lambda p: p[1])
    ax.plot([p[1] for p in fr], [p[2] for p in fr], "--", color="#888", lw=1, zorder=2)
    ax.set_xlabel("measured latency (us/inference, ESP32-S3 @240MHz)")
    ax.set_ylabel("macro-F1"); ax.set_title(ds); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

for ds in m.dataset.unique():
    d = m[m.dataset==ds]
    pts = list(zip(d.arm, d.measured_latency_us, d.macro_f1))
    fr = pareto(pts)
    print(f"{ds}")
    print("  Pareto   :", ", ".join(sorted(fr, key=lambda a: dict((l,u) for l,u,_ in pts)[a])))
    print("  DOMINATED:", ", ".join(l for l,_,_ in pts if l not in fr) or "(none)")

### 10 — Per-class F1, HATC vs MLP

In [ ]:
txt = "results/confusion_analysis.txt"
print(open(txt, encoding="utf-8").read() if have(txt) else "(run stage 4 first)")

### 11 — INT8 quantisation

In [ ]:
for d in DATASETS:
    f = f"results/qat_results_{d}.csv"
    if not have(f): print(f"(missing {f})"); continue
    print("="*80); print(LABEL[d]); print("="*80)
    t = pd.read_csv(f).groupby(["config","mode"])["macro_f1"].agg(["mean","std"]).reset_index()
    base = t[t["mode"]=="FP32"].set_index("config")["mean"]
    t["delta_vs_fp32"] = t.apply(lambda r: r["mean"] - base.get(r["config"], np.nan), axis=1)
    print(t.to_string(index=False)); print()

---
# Part C — Hardware (ESP32-S3)

Shells out to the flash scripts — a notebook cannot drive the board directly,
it needs `arduino-cli` and a real serial port. **All cells no-op safely when no
board is attached.**

### 12 — Detect the board

In [ ]:
def detect_port():
    if BOARD_PORT: return BOARD_PORT
    if not os.path.exists(CLI): return None
    r = subprocess.run([CLI,"board","list"], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if any(k in line for k in ("esp32","ESP32","CP210","CH9102","FTDI","USB")):
            tok = line.split()
            if tok and (tok[0].upper().startswith("COM") or tok[0].startswith("/dev/")):
                return tok[0]
    return None

PORT = detect_port()
print("board port:", PORT or "NOT DETECTED - Part C will skip")

### 13 — Parity + latency

The critical correctness check. Each build runs 1000 real verification flows
on-device and must reproduce the host model's predictions exactly.

Expected: **1000/1000 (100.0%)**. Before the export bias-truncation fix this
read 56.2%, so anything materially below 100% means the device is diverging
from the simulated C path.

In [ ]:
if PORT:
    run(["bash","firmware/flash_verify.sh",PORT], "flash_verify.log", timeout=1800)
else:
    print("[SKIP] no board")

### 14 — Measured latency frontier

Flashes each exported config and records median latency. This is what turns the
Pareto cost axis from *analytic MACs* into *measured microseconds*.

In [ ]:
CONFIGS = []
for d in DATASETS:
    pre = "edge" if d=="edge" else "cic25"
    CONFIGS += [c for c in [f"{pre}_hatc_d8_k8", f"{pre}_hatc_d24_k8",
                            f"{pre}_mlp_h16", f"{pre}_mlp_h32",
                            f"{pre}_mlp_h64", f"{pre}_mlp_h96"]
                if os.path.isdir(f"firmware/configs/{c}")]

rows = []
if PORT:
    for c in CONFIGS:
        rc, out = run(["bash","firmware/flash_config.sh",c,PORT],
                      f"measured_{c}.log", timeout=1200)
        rec = {"config": c}
        for line in out.splitlines():
            if "host-reproduction:" in line:
                rec["parity"] = line.split(":",1)[1].strip()
            if line.startswith("latency: median=") and "us" in line:
                rec["median_us"] = float(line.split("median=")[1].split("us")[0])
        rows.append(rec)
    mm = pd.DataFrame(rows); mm.to_csv("results/measured_latency.csv", index=False)
    display(mm)
else:
    print("[SKIP] no board — configs that would be measured:")
    for c in CONFIGS: print("   ", c)

### 15 — Energy

A single inference is ~250–800 µs while affordable USB meters sample at
1–10 Hz, so per-inference energy cannot be read directly. These builds
alternate a **silent** 15 s ACTIVE phase with a 15 s IDLE phase so a slow meter
settles in each:

$$E_\text{inference} = (P_\text{active} - P_\text{idle}) \times t_\text{inference}$$

Read the meter ~2 s into each phase. Serial stays quiet during the measured
phases because UART draw would otherwise dominate the difference.

In [ ]:
ENERGY = [c for c in ["edge_hatc_d24_k8","edge_mlp_h96",
                      "cic25_hatc_d24_k8","cic25_mlp_h32"]
          if os.path.isdir(f"firmware/configs/{c}")]
if PORT:
    for c in ENERGY:
        print(f"\n=== {c} — watch the meter, record ACTIVE then IDLE ===")
        run(["bash","firmware/flash_power.sh",c,PORT], f"power_{c}.log", timeout=900)
else:
    print("[SKIP] no board — energy configs prepared:")
    for c in ENERGY: print("   ", c)

---
## Outputs

| path | contents |
|---|---|
| `results/` | all CSVs, per-class analysis, measured latency |
| `logs/` | stdout/stderr of every stage |
| `firmware/configs/` | exported weight headers per configuration |
| `HATC-IDS_Report.docx` | design + methodology + full results |
| `CONSOLIDATED_RESULTS.md` | narrative summary |
| `MEASURED_HARDWARE_RESULTS.md` | full hardware session record |
| `BOARD_RUNBOOK.md` | step-by-step hardware procedure |

**Carry this caveat into the write-up.** The architectural ablations
(grouped-block classifier quantisation, distillation, piecewise embedding, and
two negative results) were run on TON_IoT only. They are design rationale, not
evidence, for the two datasets reported here — see Section 6 of the report.